# Пример использования `fedstat`

Библиотека скачивает данные показателей с fedstat.ru (ЕМИСС) и отдаёт нормализованный `pandas.DataFrame`.

> Нужен доступ к fedstat.ru (российский IP). Показатель в примере — **31452** «Средняя цена 1 кв. м жилья» (id из URL `https://www.fedstat.ru/indicator/31452`).

## 1. Какие фильтры есть у показателя

In [ ]:
import fedstat

fedstat.list_filters("31452").head(20)

## 2. Шаблон фильтров
`filter_template` возвращает словарь со всеми полями (значение `"*"` = все). Правим только нужное; пропущенные поля берутся целиком.

In [ ]:
f = fedstat.filter_template("31452")
f

In [ ]:
f["Год"] = [str(y) for y in range(2017, 2027)]   # диапазон лет 2017..2026
f["Рынок жилья"] = "Первичный рынок жилья"
f["Типы квартир"] = "Все типы квартир"
# ВАЖНО: для лет с 2023 РФ идёт как «без учёта новых субъектов»,
# а не «Российская Федерация» (иначе выборка пустая -> ошибка 302).
f["Классификатор объектов административно-территориального деления (ОКАТО)"] = "без учета новых субъектов"
f

## 3. Скачать данные — нормализованный («длинный») DataFrame

In [ ]:
df = fedstat.load("31452", filters=f)
print(df.shape)
df.head(10)

## 4. «Широкий» вид: годы в строках, кварталы по столбцам

In [ ]:
fedstat.to_wide(df, columns="PERIOD", values="VALUE", index="TIME")

## 5. Сохранить в CSV / Excel

In [ ]:
df.to_csv("cena.csv", index=False)
df.to_excel("cena.xlsx", index=False)
print("сохранено: cena.csv, cena.xlsx")

---
### Подсказки
- Неверное имя поля/значения -> `FilterError` с подсказкой похожего варианта.
- Ошибка **302** обычно = выбранная комбинация фильтров пустая (проверьте значения).
- Ошибка **503** = fedstat перегружен; повторите позже или увеличьте `retry_max_times`.
- `load(..., with_codes=True)` — добавить исходные коды измерений.